<!-- @format -->

# Sparse Autoencoder


In [14]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

while not (
    (PROJECT_ROOT / "data_loader.py").is_file()
    and (PROJECT_ROOT / "models" / "mlp_classifier.py").is_file()
):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate the project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: D:\Main\sem5\deep-learning\ASG_2


In [15]:
import json
import time

import matplotlib.pyplot as plt
import torch
from torch import nn, optim

from data_loader import get_dataloaders
from models.autoencoder import build_vanilla_autoencoder
from utils.metrics import psnr, ssim
from utils.paths import (
    CHECKPOINT_DIR,
    DATA_DIR,
    FIGURES_DIR,
    METRICS_DIR,
    ensure_results_dirs,
    get_device,
)
from utils.reproducibility import set_seed


In [16]:
SEED = 42
BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 0.001
LATENT_DIM = 64
SPARSITY_WEIGHT = 0.001


DEVICE = get_device()
ensure_results_dirs()

print(f"Device: {DEVICE}")


Device: xpu


In [17]:
print(f"Device: {DEVICE}")
print(f"Latent dimension: {LATENT_DIM}")
print(f"Sparsity weight: {SPARSITY_WEIGHT}")


Device: xpu
Latent dimension: 64
Sparsity weight: 0.001


In [18]:
train_loader, val_loader, test_loader = get_dataloaders(batch_size=BATCH_SIZE)

print(f"Training batches:   {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Testing batches:    {len(test_loader)}")

# Inspect one batch
images, labels = next(iter(train_loader))

print("\nBatch inspection:")
print(f"Images shape: {images.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Pixel range: [{images.min():.4f}, {images.max():.4f}]")


Training batches:   438
Validation batches: 55
Testing batches:    55

Batch inspection:
Images shape: torch.Size([128, 1, 28, 28])
Labels shape: torch.Size([128])
Pixel range: [0.0000, 1.0000]


In [ ]:
encoder, decoder = build_sparse_autoencoder(latent_dim=LATENT_DIM)

encoder = encoder.to(DEVICE)
decoder = decoder.to(DEVICE)

print("Encoder:")
print(encoder)

print("\nDecoder:")
print(decoder)

# Test forward pass
sample_images, _ = next(iter(train_loader))
sample_images = sample_images.to(DEVICE)

with torch.no_grad():
    latent = encoder(sample_images)
    reconstructed = decoder(latent)

print("\nTensor inspection:")
print(f"Input:           {sample_images.shape}")
print(f"Latent:          {latent.shape}")
print(f"Reconstruction:  {reconstructed.shape}")


Encoder:
Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=64, bias=True)
  (4): ReLU()
)

Decoder:
Sequential(
  (0): Linear(in_features=64, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=784, bias=True)
  (3): Sigmoid()
  (4): Unflatten(dim=1, unflattened_size=(1, 28, 28))
)

Tensor inspection:
Input:           torch.Size([128, 1, 28, 28])
Latent:          torch.Size([128, 64])
Reconstruction:  torch.Size([128, 1, 28, 28])
